In [3]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1SBXeTGYQrmQXCCDQz21XGJ9iKE0HM-VZP8r69P5TOZ0"
SHEET_NAME = "1.Clientlist"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()


# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [4]:
# all_varchar=true avoids cast errors when Sheets mixes text/emoji/booleans in one column
duck.sql(
    f"""
SELECT * FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
)
"""
)

┌───────────────────────┬───────────────────┬──────────────┬───────────┬────────────────────────┬────────────────────────────────────────────────┬───────────────┬──────────────────┬─────────────────────────────────────────┬──────────────────────────────┬──────────────────────────────────────────────────────────────┬────────────┬─────────────────────────┬─────────────────────────┬───────────┬──────────────┐
│ easybill_kundennummer │ last_invoice_date │ € net billed │ spe. care │    wochenliste_ids     │                 easybill_firma                 │ easybill_name │ easybill_vorname │            easybill_address             │         medisoft_ids         │                        medisoft_names                        │ sim_scores │         zoho_id         │      link to zoho       │ validated │ no_migration │
│        varchar        │      varchar      │   varchar    │  varchar  │        varchar         │                    varchar                     │    varchar    │     varchar      

In [ ]:
duck.sql(
    """
    select distinct on("Kontakt: Kundennummer") * 
    from pg.easybill.contacts
    """
)

┌─────────────────────┬───────────────────────┬────────────────────────────┬────────────────────┬─────────────────┬─────────────────────────────────┬────────────────┬────────────────────┬───────────────┬───────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────┬───────────────────────┬──────────────────────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────────────┬────────────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────────┬───────────────────────────────┬────────────────────┬────────────────────┬──────────────┬───────────────────────┬────────────────────────────────┬───────────────────────────┬───────────────────┬─────────────────┬─────────────────┬─────────────────────┬──────────────────────────────┬──────────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┬──────────────